In [1]:
"""
Build assumption_master sheet: globals
-----
Purpose: Extract truly scheme-agnostic single-value constants from
Basis and Assumptions.xlsx > Assumptions sheet into a long-format
key-value table.

Author: Mohamed Hussein Gadiwala
Basis version: A2026_07 (assumption master)
"""

from pathlib import Path
from datetime import datetime, timezone
import hashlib
import polars as pl
import openpyxl

# --- Paths (mirror basis_master structure) ---
BASE = Path(r"J:\Product & Client Solutions\Toolbox\Calculators"
            r"\Business Tools\Tender Summary\GAP Pricing Engine")

SOURCE_XLSX = BASE / "Basis and Assumptions.xlsx"
SOURCE_SHEET = "Assumptions"

ASSUMPTION_VERSION = "A2026_07"
SNAPSHOT_DIR = BASE / "assumption_snapshots" / f"assumption_{ASSUMPTION_VERSION}"
MASTER_XLSX = BASE / "assumption_master" / f"assumption_master_{ASSUMPTION_VERSION}.xlsx"
BUILD_LOG = SNAPSHOT_DIR / "build_log.csv"

SNAPSHOT_DIR.mkdir(parents=True, exist_ok=True)
MASTER_XLSX.parent.mkdir(parents=True, exist_ok=True)

SHEET_NAME = "globals"
PARQUET_PATH = SNAPSHOT_DIR / f"{SHEET_NAME}.parquet"

BUILT_AT = datetime.now(timezone.utc).isoformat(timespec="seconds")
BUILT_BY = "Mohamed Hussein Gadiwala"

print(f"Building: {SHEET_NAME}")
print(f"Source:   {SOURCE_XLSX.name} > {SOURCE_SHEET}")
print(f"Target:   {PARQUET_PATH}")
print(f"Built at: {BUILT_AT}")

Building: globals
Source:   Basis and Assumptions.xlsx > Assumptions
Target:   J:\Product & Client Solutions\Toolbox\Calculators\Business Tools\Tender Summary\GAP Pricing Engine\assumption_snapshots\assumption_A2026_07\globals.parquet
Built at: 2026-07-21T14:24:12+00:00


In [2]:
# data_only=True → get computed values, not formulas
wb = openpyxl.load_workbook(SOURCE_XLSX, data_only=True)
wb = openpyxl.load_workbook(SOURCE_XLSX, data_only=True)

ws = wb["Assumptions"]
ws_ci = wb["CI Basis"]

# Compute source SHA-256 for audit
source_bytes = SOURCE_XLSX.read_bytes()
SOURCE_SHA = hashlib.sha256(source_bytes).hexdigest()
print(f"Source SHA-256: {SOURCE_SHA[:16]}...")

# --- Locate values by explicit cell reference (no fuzzy search) ---
# These are the anchor cells I identified from the sheet.
# Each entry: (param_name, cell_ref, notes)
GLOBAL_ANCHORS = [
    # VAT — there are TWO VAT cells on the sheet.
    # The 0% one appears to be a display/formula cell.
    # The 15% one is the true assumption (row labelled "Update this cell if VAT assumption changes").
    ("vat_rate",              "C24", "Standard SA VAT rate. Update this cell if VAT assumption changes."),

    # Country / commission config
    ("active_country_code",   "C33", "1=SA, 2=NAM, 3=eSw, 4=No Commission"),
    ("namibia_competitive_factor", "E29", "Adjustment to base rates for Namibia"),
    ("namibia_stamps_duty",   "E31", "Namibia stamps duty"),
    # Namibia Covid loading factor is blank in source → preserved as null

    # Scheme size threshold
    ("scheme_size_large_threshold", "B22", "Members >= this → Large, else Small"),

    # Education product margin (small EAC block)
    ("eac_profit_margin",     "C26", "Education product profit margin"),
    ("eac_admin_adjustment",  "C1050", "EAC Admin adjustment (100.5%)"),
    ("eac_optional_cover_margin", "C1051", "EAC optional cover margin"),
    ("eac_max_cover",         "C1052", "EAC max cover ceiling"),
]

# ⚠ IMPORTANT: The cell references above are placeholders.
# Real refs will be resolved in Cell 3 by locating anchor labels.
# I've kept them here as a documentation of INTENT only.

Source SHA-256: 75835889523ee0dd...


In [3]:
def find_all_cells_any_col(ws, label: str, max_row: int = 1200, max_col: int = 30) -> list[tuple[int, int, str]]:
    """Find ALL cells matching `label` exactly, anywhere on the sheet.
    Returns list of (row, col_num, col_letter)."""
    hits = []
    for row in range(1, max_row + 1):
        for col in range(1, max_col + 1):
            v = ws.cell(row=row, column=col).value
            if isinstance(v, str) and v.strip() == label:
                hits.append((row, col, openpyxl.utils.get_column_letter(col)))
    return hits

def peek_neighbours(ws, row: int, col: int, span: int = 3) -> str:
    """Return a short display of cells to the right of (row, col)."""
    parts = []
    for dc in range(0, span + 1):
        v = ws.cell(row=row, column=col + dc).value
        letter = openpyxl.utils.get_column_letter(col + dc)
        parts.append(f"{letter}{row}={v!r}")
    return " | ".join(parts)

LABELS_TO_LOCATE = [
    "VAT",
    "Active Country",
    "Namibia - Competitive Factor",
    "Namibia - Stamps Duty",
    "Namibia - Covid Loading Factor",
    "Profit Margin Education",
    "Admin Ajustment",
    "Optional cover margin",
    "Max Cover",
    "Scheme Size",
    "0 to 50",  # anchor for scheme_fees later — free bonus check
]

print("=" * 90)
print("DIAGNOSTIC SCAN — location of every anchor label on the Assumptions sheet")
print("=" * 90)

for label in LABELS_TO_LOCATE:
    hits = find_all_cells_any_col(ws, label)
    print(f"\n'{label}': {len(hits)} match(es)")
    for i, (r, c, letter) in enumerate(hits):
        neighbours = peek_neighbours(ws, r, c, span=3)
        print(f"  [{i}] {neighbours}")

DIAGNOSTIC SCAN — location of every anchor label on the Assumptions sheet

'VAT': 3 match(es)
  [0] B31='VAT' | C31=0 | D31=None | E31=None
  [1] B33='VAT' | C33=0.15 | D33='Update this cell if VAT assumption changes' | E33=None
  [2] B263='VAT' | C263=0.15 | D263=None | E263=None

'Active Country': 1 match(es)
  [0] B44='Active Country' | C44=1 | D44=None | E44=None

'Namibia - Competitive Factor': 1 match(es)
  [0] E40='Namibia - Competitive Factor' | F40=-0.35 | G40=None | H40=None

'Namibia - Stamps Duty': 1 match(es)
  [0] E42='Namibia - Stamps Duty' | F42=0.03 | G42=None | H42=None

'Namibia - Covid Loading Factor': 1 match(es)
  [0] E41='Namibia - Covid Loading Factor' | F41=None | G41=None | H41=None

'Profit Margin Education': 1 match(es)
  [0] B35='Profit Margin Education' | C35=0.06 | D35=None | E35=None

'Admin Ajustment': 1 match(es)
  [0] B196='Admin Ajustment' | C196=1.005 | D196=None | E196=None

'Optional cover margin': 1 match(es)
  [0] B197='Optional cover margin' | 

In [4]:
def find_all_cells_any_col(ws, label: str, max_row: int = 1200, max_col: int = 30) -> list[tuple[int, int]]:
    """Find ALL cells matching `label` exactly, anywhere on the sheet.
    Returns list of (row, col_num) — 1-indexed."""
    hits = []
    for row in range(1, max_row + 1):
        for col in range(1, max_col + 1):
            v = ws.cell(row=row, column=col).value
            if isinstance(v, str) and v.strip() == label:
                hits.append((row, col))
    return hits

def value_at_offset(
    ws,
    label: str,
    col_offset: int = 1,
    row_offset: int = 0,
    occurrence: int = 0,
) -> object:
    """Get value at (row+row_offset, col+col_offset) relative to the Nth label match.
    occurrence: 0 = first, 1 = second, -1 = last."""
    hits = find_all_cells_any_col(ws, label)
    if not hits:
        return None
    row, col = hits[occurrence]
    return ws.cell(row=row + row_offset, column=col + col_offset).value


# --- Anchor list: (label, col_offset, row_offset, canonical_name, occurrence, notes) ---
ANCHORS = [
    # VAT: pin to occurrence=1 (the 15% cell with "Update this cell..." note).
    # occurrence=0 is a 0% toggle-off leftover; occurrence=2 is unrelated LSDB VAT.
    ("VAT",                              1, 0, "vat_rate",                       1,
     "SA VAT default. Source has 0%/15% toggle; pinned to 15% (B33)."),

    ("Active Country",                   1, 0, "active_country_code",            0,
     "1=SA, 2=NAM, 3=eSw, 4=No Commission (B44)"),

    # Namibia block sits in column E, not B — value in F
    ("Namibia - Competitive Factor",     1, 0, "namibia_competitive_factor",     0,
     "Adjustment to base rates for Namibia (E40 → F40)"),

    ("Namibia - Stamps Duty",            1, 0, "namibia_stamps_duty",            0,
     "Namibia stamps duty rate (E42 → F42)"),

    ("Namibia - Covid Loading Factor",   1, 0, "namibia_covid_loading_factor",   0,
     "Namibia Covid loading (blank in source A2026_07; preserved as null)"),

    ("Profit Margin Education",          1, 0, "eac_profit_margin",              0,
     "Education product profit margin (B35 → C35)"),

    ("Admin Ajustment",                  1, 0, "eac_admin_adjustment",           0,
     "EAC admin adjustment factor (source typo 'Ajustment'; B196 → C196)"),

    ("Optional cover margin",            1, 0, "eac_optional_cover_margin",      0,
     "EAC optional cover margin (B197 → C197)"),

    # Max Cover: 9 occurrences on sheet. Occurrence 0 = B198 = 20M = EAC. Confirmed.
    ("Max Cover",                        1, 0, "eac_max_cover",                  0,
     "EAC max cover (B198 → C198). Occurrence 0 of 9 Max Cover labels."),
]

rows = []
for label, col_off, row_off, canonical, occurrence, notes in ANCHORS:
    raw_val = value_at_offset(ws, label, col_offset=col_off, row_offset=row_off, occurrence=occurrence)
    rows.append({
        "param_name": canonical,
        "param_value": raw_val,
        "source_label": label,
        "source_occurrence": occurrence,
        "notes": notes,
    })

# ----------------------------------------------------------
# CI standalone loading
# Source: CI Basis!Q172
# Stored in globals so engine can use:
# ctx.globals["ci_standalone_loading"]
# ----------------------------------------------------------

rows.append({
    "param_name": "ci_standalone_loading",
    "param_value": ws_ci["Q172"].value,
    "source_label": "CI Basis!Q172",
    "source_occurrence": 0,
    "notes": "Standalone CI loading sourced from CI Basis sheet",
})

# ----------------------------------------------------------
# GLA EOH
# Source: Assumptions!E229:H231
# ----------------------------------------------------------

# GLA EOH
rows.extend([
    {
        "param_name": "eoh_amount",
        "param_value": ws["E229"].value,
        "source_label": "E229",
        "source_occurrence": 0,
        "notes": "GLA EOH base amount",
    },
    {
        "param_name": "eoh_layer1",
        "param_value": ws["F229"].value,
        "source_label": "F229",
        "source_occurrence": 0,
        "notes": "GLA EOH first layer threshold",
    },
    {
        "param_name": "eoh_layer2",
        "param_value": ws["G229"].value,
        "source_label": "G229",
        "source_occurrence": 0,
        "notes": "GLA EOH second layer threshold",
    },
    {
        "param_name": "eoh_perc1",
        "param_value": ws["F230"].value,
        "source_label": "F230",
        "source_occurrence": 0,
        "notes": "GLA EOH first layer percentage",
    },
    {
        "param_name": "eoh_perc2",
        "param_value": ws["G230"].value,
        "source_label": "G230",
        "source_occurrence": 0,
        "notes": "GLA EOH second layer percentage",
    },
    {
        "param_name": "eoh_perc3",
        "param_value": ws["H230"].value,
        "source_label": "H230",
        "source_occurrence": 0,
        "notes": "GLA EOH third layer percentage",
    },
    # RENAMED: eoh_max → eoh_layer1_cap (F231 is a per-layer cap, not overall)
    {
        "param_name": "eoh_layer1_cap",
        "param_value": ws["F231"].value,
        "source_label": "F231",
        "source_occurrence": 0,
        "notes": "GLA EOH layer 1 contribution cap (max amount from layer 1)",
    },
    # NEW: layer 2 cap
    {
        "param_name": "eoh_layer2_cap",
        "param_value": ws["G231"].value,
        "source_label": "G231",
        "source_occurrence": 0,
        "notes": "GLA EOH layer 2 contribution cap (max amount from layer 2)",
    },
])

# CI EOH
rows.extend([
    {
        "param_name": "eoh_amount_ci",
        "param_value": ws["E235"].value,
        "source_label": "E235",
        "source_occurrence": 0,
        "notes": "CI EOH base amount",
    },
    {
        "param_name": "eoh_layer1_ci",
        "param_value": ws["F235"].value,
        "source_label": "F235",
        "source_occurrence": 0,
        "notes": "CI EOH first layer threshold",
    },
    {
        "param_name": "eoh_layer2_ci",
        "param_value": ws["G235"].value,
        "source_label": "G235",
        "source_occurrence": 0,
        "notes": "CI EOH second layer threshold",
    },
    {
        "param_name": "eoh_perc1_ci",
        "param_value": ws["F236"].value,
        "source_label": "F236",
        "source_occurrence": 0,
        "notes": "CI EOH first layer percentage",
    },
    {
        "param_name": "eoh_perc2_ci",
        "param_value": ws["G236"].value,
        "source_label": "G236",
        "source_occurrence": 0,
        "notes": "CI EOH second layer percentage",
    },
    {
        "param_name": "eoh_perc3_ci",
        "param_value": ws["H236"].value,
        "source_label": "H236",
        "source_occurrence": 0,
        "notes": "CI EOH third layer percentage",
    },
    # NEW: CI per-layer caps
    {
        "param_name": "eoh_layer1_cap_ci",
        "param_value": ws["F237"].value,
        "source_label": "F237",
        "source_occurrence": 0,
        "notes": "CI EOH layer 1 contribution cap",
    },
    {
        "param_name": "eoh_layer2_cap_ci",
        "param_value": ws["G237"].value,
        "source_label": "G237",
        "source_occurrence": 0,
        "notes": "CI EOH layer 2 contribution cap",
    },
])
# Scheme size Large threshold: "Scheme Size" label at F24.
# Layout below the label:
#   F25 = 0        G25 = Small
#   F26 = 400      G26 = Large
# So threshold value = F26 = (row+2, col+0) relative to label.
hit_list = find_all_cells_any_col(ws, "Scheme Size")
if hit_list:
    r, c = hit_list[0]  # F24 → r=24, c=6
    threshold = ws.cell(row=r + 2, column=c).value
    rows.append({
        "param_name": "scheme_size_large_threshold",
        "param_value": threshold,
        "source_label": "Scheme Size > Large",
        "source_occurrence": 0,
        "notes": "Members >= this → Large scheme, else Small (F26)",
    })
else:
    raise RuntimeError("'Scheme Size' anchor not found — sheet layout has changed.")

print("--- Extracted values ---")
for row in rows:
    print(f"  {row['param_name']:40s} = {row['param_value']}")

--- Extracted values ---
  vat_rate                                 = 0.15
  active_country_code                      = 1
  namibia_competitive_factor               = -0.35
  namibia_stamps_duty                      = 0.03
  namibia_covid_loading_factor             = None
  eac_profit_margin                        = 0.06
  eac_admin_adjustment                     = 1.005
  eac_optional_cover_margin                = 0.15
  eac_max_cover                            = 20000000
  ci_standalone_loading                    = 0.2
  eoh_amount                               = 130000
  eoh_layer1                               = 10000000
  eoh_layer2                               = 20000000
  eoh_perc1                                = 0.07
  eoh_perc2                                = 0.03
  eoh_perc3                                = 0.02
  eoh_layer1_cap                           = 700000.0000000001
  eoh_layer2_cap                           = 600000
  eoh_amount_ci                            = 650

In [5]:
df = pl.DataFrame(rows)

def classify(v):
    if v is None:            return "null"
    if isinstance(v, bool):  return "bool"
    if isinstance(v, int):   return "int"
    if isinstance(v, float): return "float"
    return "string"

df = df.with_columns([
    pl.col("param_value").map_elements(
        classify,
        return_dtype=pl.Utf8,
        skip_nulls=False,          # ← the fix
    ).alias("param_type"),
    pl.col("param_value").map_elements(
        lambda v: "" if v is None else str(v),
        return_dtype=pl.Utf8,
        skip_nulls=False,          # ← the fix
    ).alias("param_value_str"),
])

df = df.select([
    "param_name",
    "param_value_str",
    "param_type",
    "source_label",
    "source_occurrence",
    "notes",
]).rename({"param_value_str": "param_value"})

# Structural checks
assert df["param_name"].n_unique() == df.height, "Duplicate param_name!"
assert not df["param_name"].is_null().any(), "Null param_name!"

# Frozen expectations for A2026_07
expected = {
    "vat_rate":                        0.15,
    "active_country_code":             1,
    "namibia_competitive_factor":     -0.35,
    "namibia_stamps_duty":             0.03,
    "eac_profit_margin":               0.06,
    "eac_admin_adjustment":            1.005,
    "eac_optional_cover_margin":       0.15,
    "eac_max_cover":                   20_000_000,
    "scheme_size_large_threshold":     400,
    "ci_standalone_loading":  0.20,

    # GLA EOH block
    "eoh_amount":       130000,
    "eoh_layer1":       10000000,
    "eoh_layer2":       20000000,
    "eoh_perc1":        0.07,
    "eoh_perc2":        0.03,
    "eoh_perc3":        0.02,
    "eoh_layer1_cap":   700000,     # renamed from eoh_max
    "eoh_layer2_cap":   600000,     # NEW

    # CI EOH block
    "eoh_amount_ci":       65000,
    "eoh_layer1_ci":       10000000,
    "eoh_layer2_ci":       20000000,
    "eoh_perc1_ci":        0.04,
    "eoh_perc2_ci":        0.02,
    "eoh_perc3_ci":        0.01,
    "eoh_layer1_cap_ci":   400000,  # NEW
    "eoh_layer2_cap_ci":   400000,  # NEW
}
for name, expected_val in expected.items():
    row = df.filter(pl.col("param_name") == name)
    assert row.height == 1, f"{name} not found in df"
    actual = float(row["param_value"][0])
    assert abs(actual - float(expected_val)) < 1e-6, (
        f"{name} mismatch: got {actual}, expected {expected_val}"
    )

# namibia_covid_loading_factor must be null (basis rule 5.5: blanks preserved)
covid = df.filter(pl.col("param_name") == "namibia_covid_loading_factor")
assert covid.height == 1
assert covid["param_type"][0] == "null", (
    f"Expected null for namibia_covid_loading_factor, got type {covid['param_type'][0]}"
)

print(f"✓ {df.height} globals extracted")
print(f"✓ All {len(expected)} value assertions passed")
print(f"✓ Namibia Covid loading preserved as null (rule 5.5)")
print()
print(df)

✓ 27 globals extracted
✓ All 26 value assertions passed
✓ Namibia Covid loading preserved as null (rule 5.5)

shape: (27, 6)
┌─────────────────┬─────────────┬────────────┬─────────────────┬─────────────────┬─────────────────┐
│ param_name      ┆ param_value ┆ param_type ┆ source_label    ┆ source_occurren ┆ notes           │
│ ---             ┆ ---         ┆ ---        ┆ ---             ┆ ce              ┆ ---             │
│ str             ┆ str         ┆ str        ┆ str             ┆ ---             ┆ str             │
│                 ┆             ┆            ┆                 ┆ i64             ┆                 │
╞═════════════════╪═════════════╪════════════╪═════════════════╪═════════════════╪═════════════════╡
│ vat_rate        ┆ 0.15        ┆ float      ┆ VAT             ┆ 1               ┆ SA VAT default. │
│                 ┆             ┆            ┆                 ┆                 ┆ Source has 0%/… │
│ active_country_ ┆ 1.0         ┆ float      ┆ Active Country  ┆ 0 

In [6]:
df.write_parquet(PARQUET_PATH)
parquet_bytes = PARQUET_PATH.read_bytes()
PARQUET_SHA = hashlib.sha256(parquet_bytes).hexdigest()
PARQUET_SIZE_KB = len(parquet_bytes) / 1024

print(f"✓ Wrote {PARQUET_PATH.name}")
print(f"  Size:    {PARQUET_SIZE_KB:.2f} KB")
print(f"  SHA-256: {PARQUET_SHA[:16]}...")

if MASTER_XLSX.exists():
    master_wb = openpyxl.load_workbook(MASTER_XLSX)
else:
    master_wb = openpyxl.Workbook()
    if "Sheet" in master_wb.sheetnames:
        del master_wb["Sheet"]

if SHEET_NAME in master_wb.sheetnames:
    del master_wb[SHEET_NAME]

ws_out = master_wb.create_sheet(SHEET_NAME)
ws_out.append(df.columns)
for row in df.iter_rows():
    ws_out.append(list(row))

master_wb.save(MASTER_XLSX)
print(f"✓ Master workbook updated: {MASTER_XLSX.name}")

✓ Wrote globals.parquet
  Size:    3.86 KB
  SHA-256: e27ad2b4abfd3f6c...
✓ Master workbook updated: assumption_master_A2026_07.xlsx


In [7]:
import csv

LOG_HEADERS = [
    "built_at", "built_by", "sheet", "rows", "cols",
    "parquet_path", "parquet_size_kb", "parquet_sha256",
    "source_file", "source_sheet", "source_sha256",
    "assumption_version", "notes",
]

log_row = {
    "built_at": BUILT_AT,
    "built_by": BUILT_BY,
    "sheet": SHEET_NAME,
    "rows": df.height,
    "cols": len(df.columns),
    "parquet_path": str(PARQUET_PATH.relative_to(BASE)),
    "parquet_size_kb": f"{PARQUET_SIZE_KB:.2f}",
    "parquet_sha256": PARQUET_SHA,
    "source_file": SOURCE_XLSX.name,
    "source_sheet": SOURCE_SHEET,
    "source_sha256": SOURCE_SHA,
    "assumption_version": ASSUMPTION_VERSION,
    "notes": "First assumption master sheet. Anchor-based extraction with diagnostic-verified coordinates. VAT pinned to occurrence=1 (15%).",
}

write_header = not BUILD_LOG.exists()
with BUILD_LOG.open("a", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=LOG_HEADERS)
    if write_header:
        writer.writeheader()
    writer.writerow(log_row)

print(f"✓ build_log.csv updated ({BUILD_LOG.name})")
print(f"\n{'='*60}")
print(f"BUILD COMPLETE: {SHEET_NAME}")
print(f"{'='*60}")

✓ build_log.csv updated (build_log.csv)

BUILD COMPLETE: globals
